# Simple RAG Agent — Recursive Chunking


In [ ]:
!pip install -q sentence-transformers groq numpy



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import re
import textwrap
from dataclasses import dataclass, field
import numpy as np
from sentence_transformers import SentenceTransformer
from groq import Groq

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")  # set this env var before running
LLM_MODEL = "openai/gpt-oss-20b"                    # any Groq-hosted chat model works

llm_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
if llm_client is None:
    print("No GROQ_API_KEY set — retrieval will still work below, but generate_answer()\n"
          "will fall back to returning the best-matching chunk instead of a generated answer.")
else:
    print("Setup complete.")


Setup complete.


## 1. The source document

Replace `DOCUMENT` with your own text — load it from a `.txt`/`.pdf`/scraped page, it doesn't matter,
as long as it ends up as one big string.


In [3]:
DOCUMENT = """Robotics and Industrial Automation

Introduction

Robotics combines mechanical design, electronics, and software to build machines that sense,
plan, and act in the physical world. Modern industrial robots are used for welding, assembly,
and material handling, while service robots increasingly work alongside people in homes,
hospitals, and warehouses.

Perception

A robot perceives its environment through sensors such as cameras, LiDAR, ultrasonic
rangefinders, and inertial measurement units. Raw sensor data is noisy, so perception
pipelines typically filter, fuse, and interpret it before the robot acts on it. Computer
vision models can detect objects, estimate depth, and track motion in real time.

Control and Actuation

Once a robot understands its surroundings, a control system decides how to move. This ranges
from simple PID loops that keep a single joint at a target angle to full-body motion planners
that coordinate dozens of joints while avoiding obstacles. Actuators, usually electric motors
or hydraulic cylinders, convert control signals into physical motion.

Software Architecture

Most robotics software is organized around a middleware framework such as ROS (Robot
Operating System), which lets independent modules for perception, planning, and control
communicate over a shared messaging layer. This modularity makes it easier to swap in new
sensors or algorithms without rewriting the entire stack.

Challenges

Real-world robotics remains hard because physical environments are unpredictable: lighting
changes, objects move, and hardware wears down over time. Safety is a first-class concern
whenever robots operate near people, requiring redundant sensing and conservative control
limits.
"""

print(f"Document length: {len(DOCUMENT)} characters")


Document length: 1719 characters


## 2. Recursive chunking



In [4]:
@dataclass
class Chunk:
    """A single chunk of the source document."""
    text: str
    index: int
    meta: dict = field(default_factory=dict)


def recursive_split(text, chunk_size=500, separators=("\n\n", "\n", ". ", " ")):
    """Split `text` into pieces of at most `chunk_size` characters using a cascade of
    separators, from coarsest to finest (paragraphs -> lines -> sentences -> words).

    At each level: split on the current separator, then greedily re-merge the resulting
    pieces back together (up to chunk_size). Any merged piece still over chunk_size is
    recursively re-split with the next, finer separator. If we run out of separators
    entirely, fall back to a hard character cut so we always terminate.
    """
    text = text.strip()
    if not text:
        return []
    if len(text) <= chunk_size:
        return [text]
    if not separators:
        return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

    sep, remaining_seps = separators[0], separators[1:]
    pieces = [p for p in text.split(sep) if p.strip()]

    if len(pieces) == 1:
        return recursive_split(text, chunk_size, remaining_seps)

    chunks, buffer = [], ""
    for piece in pieces:
        candidate = f"{buffer}{sep}{piece}" if buffer else piece
        if len(candidate) <= chunk_size:
            buffer = candidate
        else:
            if buffer:
                chunks.extend(
                    recursive_split(buffer, chunk_size, remaining_seps)
                    if len(buffer) > chunk_size else [buffer]
                )
            buffer = piece
    if buffer:
        chunks.extend(
            recursive_split(buffer, chunk_size, remaining_seps)
            if len(buffer) > chunk_size else [buffer]
        )
    return chunks


def build_chunks(text, chunk_size=500):
    return [Chunk(t, i) for i, t in enumerate(recursive_split(text, chunk_size))]


chunks = build_chunks(DOCUMENT, chunk_size=400)
print(f"Split into {len(chunks)} chunks (sizes: {[len(c.text) for c in chunks]})")
print()
print(textwrap.shorten(chunks[0].text.replace(chr(10), ' '), width=160, placeholder=' ...'))


Split into 5 chunks (sizes: [364, 363, 373, 332, 278])

Robotics and Industrial Automation Introduction Robotics combines mechanical design, electronics, and software to build machines that sense, plan, and act ...


## 3. Embed the chunks



In [5]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def embed_texts(texts):
    # normalize_embeddings=True means a plain dot product between two rows
    # equals cosine similarity, which is all `retrieve()` needs below.
    return embedder.encode(texts, normalize_embeddings=True, show_progress_bar=False)

chunk_embeddings = embed_texts([c.text for c in chunks])
print(f"Embedding matrix shape: {chunk_embeddings.shape}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding matrix shape: (5, 384)


## 4. Retrieve

Embed the query the same way, then rank chunks by cosine similarity.


In [6]:
def retrieve(query, k=3):
    """Return the top-k (Chunk, score) pairs most similar to `query`."""
    query_emb = embed_texts([query])[0]
    scores = chunk_embeddings @ query_emb
    top_idx = np.argsort(-scores)[:k]
    return [(chunks[i], float(scores[i])) for i in top_idx]


for chunk, score in retrieve("How do robots sense their environment?"):
    preview = textwrap.shorten(chunk.text.replace(chr(10), ' '), width=110, placeholder=' ...')
    print(f"[{score:.3f}] {preview}")


[0.693] A robot perceives its environment through sensors such as cameras, LiDAR, ultrasonic rangefinders, and ...
[0.493] Once a robot understands its surroundings, a control system decides how to move. This ranges from simple ...
[0.485] Real-world robotics remains hard because physical environments are unpredictable: lighting changes, ...


## 5. Generate — turn retrieved chunks into an answer




In [10]:
SYSTEM_PROMPT = (
    "You are a helpful assistant. Answer the user's question using ONLY the context "
    "provided below. If the answer isn't in the context, say you don't know instead "
    "of guessing."
)


def generate_answer(query, retrieved, model=LLM_MODEL):
    context = "\n\n".join(f"[{i+1}] {chunk.text}" for i, (chunk, _) in enumerate(retrieved))
    if llm_client is None:
        return f"(no LLM configured) Best matching passage:\n{retrieved[0][0].text}"

    prompt = f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    response = llm_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content


## 6. Put it together: the RAG agent


In [8]:
def rag_agent(query, k=3, verbose=True):
    """Full pipeline: retrieve relevant chunks, then generate a grounded answer."""
    retrieved = retrieve(query, k=k)
    answer = generate_answer(query, retrieved)
    if verbose:
        print(f"Q: {query}\n")
        print(f"A: {answer}\n")
        print("Sources:")
        for rank, (chunk, score) in enumerate(retrieved, 1):
            preview = textwrap.shorten(chunk.text.replace(chr(10), ' '), width=100, placeholder=' ...')
            print(f"  [{rank}] score={score:.3f} - {preview}")
    return answer, retrieved


## 7. Try it


In [9]:
for q in [
    "What are the main components of a robotic system?",
    "How do robots decide how to move?",
    "Why is real-world robotics difficult?",
]:
    rag_agent(q)
    print("=" * 80)


Q: What are the main components of a robotic system?

A: The main components of a robotic system are:

1. **Perception** – sensors (cameras, LiDAR, ultrasonic rangefinders, IMUs, etc.) that gather data about the environment, followed by filtering, fusion, and interpretation to produce usable information.

2. **Control and Actuation** – a control system (from simple PID loops to full‑body motion planners) that decides how the robot should move, and actuators (electric motors, hydraulic cylinders, etc.) that convert those control signals into physical motion.

3. **Mechanical Design** – the physical structure that houses the sensors, electronics, and actuators, enabling the robot to interact with the world.

Sources:
  [1] score=0.606 - Once a robot understands its surroundings, a control system decides how to move. This ranges ...
  [2] score=0.516 - Robotics and Industrial Automation Introduction Robotics combines mechanical design, ...
  [3] score=0.510 - A robot perceives its environ